# 02 — Validation (DCC process **V1**)

**SEA-FORWARD** · OPERA Capacity Development · OceanPrediction-A toolkit

This notebook is the **manual / visual half** of Step 4.2 in the SEA-FORWARD
operational workflow. The automated half, `sftools/run_validation.py`
(Step 4.1), writes `validation_report.json`/`.txt` and `taylor_diagram.png`
per cycle -- both halves now go through the exact same statistics engine,
`sftools.validation_godae` (see Section 4), so they can never silently
disagree. This notebook implements Component **Validation Module** (DCC
process **V1**) as a Jupyter Notebook, for a single FORECAST cycle: bias
maps, scatter plots, Taylor diagrams, time series, satellite SST (Section
7), and (Section 8) optional in-situ / class-4 scoring.```

**V1 -- Validation & Model Intercomparison**: statistical comparison of model
output against three independent reference sources -- the Copernicus Marine
Forecast (Mercator anfc), satellite SST (OSTIA/ODYSSEA), and CMEMS in-situ
observations -- RMSE, bias, spatial correlation, Taylor-diagram skill
metrics. A model that fails V1 should not be trusted for the downstream D1
applications (visualisation, exercises, sensitivity analysis) in the other
three notebooks.

> **OceanPrediction-A** is the *simplest* DCC Architecture blueprint: a
> single deterministic run, no data assimilation. That's exactly why V1
> matters here -- with no assimilation step correcting the model against
> observations as it runs, all of the quality control happens *after* the
> fact, in this notebook and in `run_validation.py`.

This notebook uses CROCO forecast output already on disk for the cycle you pick
(Section 1), plus the reference sources below, which this notebook
downloads itself if not already present (Section 1c) -- each
availability-guarded so an unreachable/un-entitled CMEMS product is
skipped, not a hard failure.

**Reference datasets.**
- (i) Copernicus Marine Forecast -- `GLOBAL_ANALYSISFORECAST_PHY_001_024` /
  `cmems_mod_glo_phy_anfc_0.083deg_PT1H-m`, the operational Mercator
  analysis-forecast CROCO is downscaled from.
- (ii) Satellite SST -- OSTIA (L4) and ODYSSEA (L3S), independent products.
- (ii-b) Satellite SSS -- SMOS (L4).
- (iii) CMEMS In-Situ TAC (`INSITU_GLO_PHYBGCWAV_DISCRETE_MYNRT_013_030`,
  DOI 10.48670/moi-00036) -- trajectories and profiles,
  `sftools.validation_godae.validate_against_insitu` extends the same
  statistics to point/profile comparisons per GODAE depth layer.

**About 10 min to validate a single cycle.**


In [1]:
# ----------------------------------------------------------------------
# Setup -- run from the notebooks/ folder so sftools imports (see sftools/README.md)
# ----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import sftools.postprocess as pp
import sftools.validation as val   # every validation function (maps, profiles,
                                   # timeseries, satellite, in-situ, HTML summary)
                                   # lives in this one module
import sftools.plotting as pl
from sftools.download import cmems

import _paths


In [2]:
# ----------------------------------------------------------------------
# Forecast cycle to validate. Every forecast run lives in a cycle directory
# named YYYYMMDD (e.g. "20260711" for the 5-day cycle 11-15 July 2026),
# sibling to every other cycle under <MAIN_DIR>/<CONFIG>/. There can be
# several cycles on disk at once -- pick the one you want with CYCLE below
# (or set the SEAFORWARD_CYCLE environment variable). This notebook
# validates FORECAST cycles only, against the Copernicus Marine Forecast
# (Mercator anfc); it does not handle hindcasts.
# ----------------------------------------------------------------------
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR", "~/seaforward/forecast/model-runs"))

AVAILABLE_CYCLES = _paths.list_cycles(MAIN_DIR, CONFIG)
print(f"Forecast cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")


Forecast cycles found under /home/lell/seaforward/forecast/model-runs/Canary_12: ['20260711', '20260723', '20260729']


In [51]:
# >>> SET THIS to the cycle you want to validate, e.g. "20260711" <<<
CYCLE = os.environ.get("SEAFORWARD_CYCLE", "20260711")

CROCO_HIS, REFERENCE, MAIN_DIR = _paths.get_paths(cycle=CYCLE, config=CONFIG, main_dir=MAIN_DIR)
YORIG = 2000   # if not working, set to >>> None <<< since real CROCO output carries proper CF time units 

# >>> Depth level for the grid comparisons in Section 2 (SST/currents/SSS maps) <<<
# None -> surface (SST/SSS/surface currents). Set e.g. DEPTH_M = 100 to compare
# temperature/salinity/currents at 100 m depth instead. SSH has no depth
# dimension, so Section 2's SSH comparison always stays at the surface
# regardless of this setting.
DEPTH_M = None

# >>> Point(s) for the vertical profile overlay in Section 2b <<<
# None -> one auto-picked coastal-ish point (75% across the grid, mid-latitude).
# Or set a list of (lon, lat) tuples to profile specific locations instead, e.g.
# PROFILE_POINTS = [(-16.23, 28.10), (-15.90, 27.55)]
PROFILE_POINTS = None

# All figures/stats from this notebook are written here: a directory named
# "validation_<CYCLE>", created as a SIBLING of the cycle directories inside
# MAIN_DIR/CONFIG (not nested inside the fcst run being validated).
VALIDATION_DIR = _paths.get_validation_dir(MAIN_DIR, CONFIG, CYCLE)
print(f"Validating cycle {CYCLE}")
print(f"  CROCO history      : {CROCO_HIS}")
print(f"  Forecast reference : {REFERENCE}  ({'found' if os.path.exists(REFERENCE) else 'not downloaded yet - see Section 1c'})")
print(f"  Validation outputs : {VALIDATION_DIR}")
print(f"  Depth level (Sec 2): {'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'}")


Validating cycle 20260711
  CROCO history      : /home/lell/seaforward/forecast/model-runs/Canary_12/20260711/fcst/CROCO_FILES/croco_his.nc
  Forecast reference : /home/lell/seaforward/forecast/model-runs/Canary_12/20260711/downloaded_data/MERCATOR/MERCATOR_20260711_00.nc  (found)
  Validation outputs : /home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260711
  Depth level (Sec 2): surface


## 1. Load model output and check the run

Before comparing anything, a quick look at what actually came out of C1
(the CROCO run): grid size, time coverage, and Forecasting Accuracy
Validation Criterion **"Numerical stability -- No NaN or overflow in any
output field"** (see Technical Specification Section 9.3).


In [52]:
ds = pp.open_history(CROCO_HIS, Yorig=YORIG)

print(f"grid          : {ds.sizes['eta_rho']} x {ds.sizes['xi_rho']}  "
      f"({ds.sizes.get('s_rho', '?')} sigma levels)")
print(f"time steps    : {ds.sizes['time']}")
print(f"time coverage : {pp.times(ds)[0]}  ->  {pp.times(ds)[-1]}")

stability_ok = True
for var in ('zeta', 'temp', 'salt', 'u', 'v'):
    vals = ds[var].values
    n_nan = int(np.isnan(vals).sum())
    n_inf = int(np.isinf(vals).sum())
    ok = (n_nan == 0) and (n_inf == 0)
    stability_ok &= ok
    print(f"  {var:5s}: {'OK  ' if ok else 'FAIL'}  (NaN={n_nan}, Inf={n_inf})")

print()
print(f"Numerical stability (FR pass criterion): {'PASS' if stability_ok else 'FAIL'}")


grid          : 123 x 81  (50 sigma levels)
time steps    : 21
time coverage : 2026-07-11T00:00:00  ->  2026-07-16T00:00:00
  zeta : OK    (NaN=0, Inf=0)
  temp : OK    (NaN=0, Inf=0)
  salt : OK    (NaN=0, Inf=0)
  u    : OK    (NaN=0, Inf=0)
  v    : OK    (NaN=0, Inf=0)

Numerical stability (FR pass criterion): PASS


## 1b. Reference-product availability

Before comparing anything, check on the Copernicus Marine platform which of
the three reference sources used below are actually reachable right now:
the Global Analysis & Forecast physics product (i), the two satellite SST
products (ii). This is a metadata-only
check (`copernicusmarine describe`, no download) via
`sftools.download.cmems.dataset_available`.

Any comparison below whose product is unavailable is **skipped, not
failed** -- a temporary CMEMS outage, an un-entitled product, or missing
`copernicusmarine` credentials is not a V1 validation failure, just nothing
to compare against this run.


In [53]:
AVAIL = {
    name: cmems.dataset_available(name)
    for name in ("mercator_forecast", "ostia_l4", "odyssea_l3s", "smos_l4_sss")
}

Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:07<00:00,  3.70s/it]


  CMEMS product 'mercator_forecast' (cmems_mod_glo_phy_anfc_0.083deg_P1D-m): available


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:09<00:09,  9.27s/it]

  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available



Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:18<00:00,  9.23s/it]
                                                                                   
Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:08<00:00,  4.11s/it]


  CMEMS product 'odyssea_l3s' (IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE): available


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:09<00:09,  9.44s/it]

  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


## 1c. Download the reference products for this cycle

Every comparison below needs its reference data downloaded first -- this
notebook does that itself, into this cycle's `downloaded_data/` folder,
rather than assuming a separate pipeline step already fetched it. Each
download is availability-guarded (Section 1b): a product that's
unreachable/un-entitled right now is skipped here, and every section below
that depends on it skips too, printing why, rather than failing.

- **(i) Copernicus Marine Forecast** -- one combined file
  (`MERCATOR_<cycle>_00.nc`, thetao/so/uo/vo/zos) for the whole cycle window.
- **(ii-a) Satellite SST** -- OSTIA and ODYSSEA, one file per day.
- **(ii-b) Satellite SSS** -- SMOS, one file per day.

Files already on disk are reused, not re-downloaded (safe to re-run this
cell).


In [54]:
from datetime import datetime, timedelta

clon_full, clat_full, _ = pp.lonlatmask(ds)
DOMAIN = (float(np.nanmin(clon_full)), float(np.nanmax(clon_full)),
         float(np.nanmin(clat_full)), float(np.nanmax(clat_full)))
model_times = pd.to_datetime(pp.times(ds))
START_DATE = model_times[0].to_pydatetime()
END_DATE = model_times[-1].to_pydatetime()
CYCLE_DATE = datetime.strptime(CYCLE, "%Y%m%d")
print(f"Domain: {DOMAIN}")
print(f"Cycle window: {START_DATE} -> {END_DATE}")

# every distinct calendar day covered by this forecast cycle -- Sections 2, 3
# and 4 below each save one figure PER DAY (matching the Section 7 satellite
# comparisons), rather than one figure for just the cycle's last time step.
CYCLE_DAYS = sorted(set(model_times.strftime('%Y-%m-%d')))
print(f"Domain: {DOMAIN}")
print(f"Cycle window: {START_DATE} -> {END_DATE}")

# ---- (i) Copernicus Marine Forecast (Mercator anfc), combined reference file ----
print("\n-- Copernicus Marine Forecast --")
# Daily-mean (P1D-m), not hourly -- the hourly product is far heavier and
# isn't needed here. max_step_hours=30: an existing file's median time step
# should be ~24h (daily) -- anything much coarser means it's a stale/broken
# file, and must be re-downloaded even though its date range looks fine.
if not AVAIL['mercator_forecast']:
    print("unavailable on the CMEMS platform - Sections 2/2b will be skipped.")
elif cmems.netcdf_covers_time_range(REFERENCE, START_DATE, END_DATE, max_step_hours=30):
    print(f"already downloaded and covers the full cycle window at the expected "
         f"resolution: {REFERENCE}")
else:
    if os.path.exists(REFERENCE):
        print(f"{REFERENCE} exists but either doesn't cover the full cycle window "
             f"({START_DATE.date()} -> {END_DATE.date()}) or is at a coarser "
             f"resolution than expected -- re-downloading.")
        os.remove(REFERENCE)
    mercator_dir = os.path.dirname(REFERENCE)
    fdays = max((END_DATE.date() - CYCLE_DATE.date()).days, 0)
    cmems.download_mercator_ops(DOMAIN, CYCLE_DATE, hdays=0, fdays=fdays, outputDir=mercator_dir)
    if cmems.netcdf_covers_time_range(REFERENCE, START_DATE, END_DATE, max_step_hours=30):
        print(f"downloaded -> {REFERENCE}")
    elif os.path.exists(REFERENCE):
        print(f"download ran but {REFERENCE} still doesn't cover the full cycle window "
             f"at the expected resolution -- one or more variable downloads may have "
             f"failed; check the log above.")
    else:
        print(f"download ran but {REFERENCE} wasn't produced - check {mercator_dir} for the actual filename.")

# ---- (ii) Satellite SST: OSTIA & ODYSSEA, one file per day ----
print("\n-- Satellite SST --")
SAT_FILES = {}
for product in ("OSTIA", "ODYSSEA"):
    sat_dir = _paths.satellite_dir(MAIN_DIR, CONFIG, CYCLE, product)
    SAT_FILES[product] = cmems.download_satellite_sst(product, DOMAIN, START_DATE, END_DATE, sat_dir)

# ---- (ii-b) Satellite SSS: SMOS L4, one file per day ----
print("\n-- Satellite SSS (SMOS) --")
if not AVAIL['smos_l4_sss']:
    print("unavailable on the CMEMS platform - Section 7b will be skipped.")
    SAT_FILES['SMOS'] = {}
else:
    smos_dir = _paths.satellite_dir(MAIN_DIR, CONFIG, CYCLE, "SMOS")
    SAT_FILES['SMOS'] = cmems.download_satellite_sst("SMOS", DOMAIN, START_DATE, END_DATE, smos_dir)


Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:09<00:00,  4.72s/it]


Domain: (-22.152978897094727, -15.34702205657959, 13.937745094299316, 24.041303634643555)
Cycle window: 2026-07-11 00:00:00 -> 2026-07-16 00:00:00
Domain: (-22.152978897094727, -15.34702205657959, 13.937745094299316, 24.041303634643555)
Cycle window: 2026-07-11 00:00:00 -> 2026-07-16 00:00:00

-- Copernicus Marine Forecast --
already downloaded and covers the full cycle window at the expected resolution: /home/lell/seaforward/forecast/model-runs/Canary_12/20260711/downloaded_data/MERCATOR/MERCATOR_20260711_00.nc

-- Satellite SST --


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:08<00:08,  8.29s/it]INFO - 2026-09-05T09:03:56Z - Checking if credentials are valid.


  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available


INFO - 2026-09-05T09:04:01Z - Valid credentials from configuration file.
Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:13<00:00,  6.70s/it]


...

CMEMS: already logged in.
  [ODYSSEA] 2026-07-11: already downloaded - 2026-07-11.nc
  [ODYSSEA] 2026-07-12: already downloaded - 2026-07-12.nc
  [ODYSSEA] 2026-07-13: already downloaded - 2026-07-13.nc
  [ODYSSEA] 2026-07-14: already downloaded - 2026-07-14.nc
  [ODYSSEA] 2026-07-15: already downloaded - 2026-07-15.nc
  [ODYSSEA] 2026-07-16: already downloaded - 2026-07-16.nc

-- Satellite SSS (SMOS) --


Fetching catalogue 1:  50%|█████████████             | 1/2 [00:08<00:08,  8.29s/it]INFO - 2026-09-05T09:04:29Z - Checking if credentials are valid.


  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


INFO - 2026-09-05T09:04:33Z - Valid credentials from configuration file.
Fetching catalogue 1: 100%|██████████████████████████| 2/2 [00:12<00:00,  6.18s/it]

CMEMS: already logged in.
  [SMOS] 2026-07-11: already downloaded - 2026-07-11.nc
  [SMOS] 2026-07-12: already downloaded - 2026-07-12.nc
  [SMOS] 2026-07-13: already downloaded - 2026-07-13.nc
  [SMOS] 2026-07-14: already downloaded - 2026-07-14.nc
  [SMOS] 2026-07-15: already downloaded - 2026-07-15.nc
  [SMOS] 2026-07-16: already downloaded - 2026-07-16.nc


## 2. Bias maps -- CROCO forecast vs Copernicus Marine Forecast (i)

Three-panel maps (CROCO | reference | difference) plus domain-averaged
statistics, for the three headline Forecasting Accuracy Validation
Criteria variables (SST, SSH, surface currents) using
`sftools.validation` (see that module's docstring for the regridding
method).

**Reference product:** Global Ocean Analysis & Forecast physics, `GLOBAL_ANALYSISFORECAST_PHY_001_024` / `cmems_mod_glo_phy_anfc_0.083deg_PT1H-m` -- the operational Mercator analysis-forecast the CROCO domain is downscaled from. Skipped (not failed) if `AVAIL['mercator_forecast']` is False or `REFERENCE` isn't reachable locally.


In [55]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    sst_stats_by_day = {}
    for day in CYCLE_DAYS:
        _, sst_stats_by_day[day] = val.compare_sst(
            CROCO_HIS, REFERENCE, date=day, Yorig=YORIG, depth_m=DEPTH_M, all_dates=CYCLE_DAYS,
            out=os.path.join(VALIDATION_DIR, f'sst_vs_forecast_{day}.png'))
    sst_stats = sst_stats_by_day[CYCLE_DAYS[-1]]   # last day, kept for anything downstream expecting a single value
    print(f"-> {len(sst_stats_by_day)} figure(s) written, one per day: {CYCLE_DAYS}")
    # pass/fail against this and every other Section 9.3 criterion is decided
    # once, from the shared GODAE scorecard, in Section 6 below.
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")

SST  CROCO vs parent:
  [SST]  n=8209  bias=+0.033  RMSE=0.313  cRMSE=0.312  corr=0.989
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.051  RMSE=0.373  cRMSE=0.370  corr=0.986
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.066  RMSE=0.404  cRMSE=0.399  corr=0.981
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.069  RMSE=0.425  cRMSE=0.420  corr=0.979
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.125  RMSE=0.489  cRMSE=0.473  corr=0.975
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.519  RMSE=0.761  cRMSE=0.557  corr=0.968
-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


In [56]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    # SSH has no depth dimension -- always surface, regardless of DEPTH_M.
    ssh_stats_by_day = {}
    for day in CYCLE_DAYS:
        _, ssh_stats_by_day[day] = val.compare_ssh(
            CROCO_HIS, REFERENCE, date=day, Yorig=YORIG, all_dates=CYCLE_DAYS,
            out=os.path.join(VALIDATION_DIR, f'ssh_vs_forecast_{day}.png'))
    ssh_stats = ssh_stats_by_day[CYCLE_DAYS[-1]]
    print(f"-> {len(ssh_stats_by_day)} figure(s) written, one per day: {CYCLE_DAYS}")
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")

SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.011  cRMSE=0.011  corr=0.983
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.011  cRMSE=0.011  corr=0.983
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.011  cRMSE=0.011  corr=0.981
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.013  cRMSE=0.013  corr=0.975
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.014  cRMSE=0.014  corr=0.971
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.001  RMSE=0.015  cRMSE=0.015  corr=0.966
-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


In [57]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    cur_stats_by_day = {}
    for day in CYCLE_DAYS:
        _, cur_stats_by_day[day] = val.compare_currents(
            CROCO_HIS, REFERENCE, date=day, Yorig=YORIG, depth_m=DEPTH_M, all_dates=CYCLE_DAYS,
            out=os.path.join(VALIDATION_DIR, f'currents_vs_forecast_{day}.png'))
    cur_stats = cur_stats_by_day[CYCLE_DAYS[-1]]
    print(f"-> {len(cur_stats_by_day)} figure(s) written, one per day: {CYCLE_DAYS}")
    print()
    print("Surface velocities pass criterion is QUALITATIVE (visual consistency with")
    print("expected gyre/coastal-jet circulation) -- inspect the vector maps")
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")

speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.061  RMSE=0.106  cRMSE=0.087  corr=0.739
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.079  RMSE=0.137  cRMSE=0.112  corr=0.561
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.023  RMSE=0.087  cRMSE=0.084  corr=0.656
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.009  RMSE=0.076  cRMSE=0.076  corr=0.770
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.039  RMSE=0.110  cRMSE=0.103  corr=0.624
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.008  RMSE=0.116  cRMSE=0.116  corr=0.504
-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']

Surface velocities pass criterion is QUALITATIVE (visual consistency with
expected gyre/coastal-jet circulation) -- inspect the vector maps


In [58]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    salt_stats_by_day = {}
    for day in CYCLE_DAYS:
        _, salt_stats_by_day[day] = val.compare_sss(
            CROCO_HIS, REFERENCE, date=day, Yorig=YORIG, depth_m=DEPTH_M, all_dates=CYCLE_DAYS,
            out=os.path.join(VALIDATION_DIR, f'sss_vs_forecast_{day}.png'))
    salt_stats = salt_stats_by_day[CYCLE_DAYS[-1]]
    print(f"-> {len(salt_stats_by_day)} figure(s) written, one per day: {CYCLE_DAYS}")
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")

SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.015  RMSE=0.087  cRMSE=0.086  corr=0.969
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.018  RMSE=0.098  cRMSE=0.097  corr=0.959
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.018  RMSE=0.112  cRMSE=0.110  corr=0.945
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.022  RMSE=0.133  cRMSE=0.132  corr=0.924
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.025  RMSE=0.145  cRMSE=0.143  corr=0.910
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.023  RMSE=0.151  cRMSE=0.149  corr=0.902
-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


## 2b. Vertical profile & error-vs-depth -- CROCO forecast vs Copernicus Marine Forecast (i)

Maps (Section 2) show the horizontal pattern at one level; a vertical
profile at a point and an error-vs-depth sweep show the *vertical*
structure of the agreement -- typically largest disagreement at the
surface (wind- and mesoscale-driven) decreasing with depth (slower,
larger-scale, more geostrophic flow). Uses `sftools.validation.
compare_profile` (overlay) and `sftools.validation.error_vs_depth` (skill
vs depth).


In [59]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    clon2, clat2, _ = pp.lonlatmask(ds)

    # PROFILE_POINTS (set in Section 0) -- or fall back to one auto-picked
    # coastal-ish point (75% across the grid, mid-latitude) if not set.
    if PROFILE_POINTS:
        points = list(PROFILE_POINTS)
    else:
        j0p = ds.sizes['eta_rho'] // 2
        i0p = int(0.75 * ds.sizes['xi_rho'])
        points = [(float(clon2[j0p, i0p]), float(clat2[j0p, i0p]))]

    # one profile PER DAY in the cycle, per point, per variable
    profile_days = sorted(set(pd.to_datetime(pp.times(ds)).strftime('%Y-%m-%d')))
    print(f"Profiling {len(points)} point(s) x {len(profile_days)} day(s): {points}")

    for lon0p, lat0p in points:
        for day in profile_days:
            for var in ('temp', 'salt', 'speed'):
                out_png = os.path.join(
                    VALIDATION_DIR,
                    f'profile_{var}_vs_forecast_{lon0p:.2f}_{lat0p:.2f}_{day}.png')
                val.compare_profile(CROCO_HIS, REFERENCE, var, lon0p, lat0p, date=day,
                                    Yorig=YORIG, out=out_png)

    _, depth_stats = val.error_vs_depth(CROCO_HIS, REFERENCE, field='temp',
                                        depths=(0, 50, 100, 200, 500), Yorig=YORIG)
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping vertical profile/error-vs-depth.")


Profiling 1 point(s) x 6 day(s): [(-17.105697631835938, 18.99273681640625)]


## 2b-bis. Full-domain vertical profile (spatial spread)

Companion to Section 2b's point profile: instead of one grid cell, CROCO
and parent are averaged over the WHOLE domain at each depth level, with
+/- 1 spatial std shown as fill_between, for temp/salt/speed, one day
at a time.

you can also draw full-domain vertical profile. But this will show up important difference 
between CROCO and reference. A full-domain mean vertical profile averages over every horizontal point at each depth, 
cumulating localized errors at every level. The profile shape then shows clear difference betwwen CROCO and reference

In [60]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    for day in CYCLE_DAYS:
        for var in ('temp', 'salt', 'speed'):
            out_png = os.path.join(VALIDATION_DIR, f'profile_domain_{var}_vs_forecast_{day}.png')
            val.domain_profile(CROCO_HIS, REFERENCE, var, day, Yorig=YORIG, out=out_png)
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping full-domain vertical profile.")

## 2c. Depth-resolved comparison -- CROCO forecast vs Copernicus Marine Forecast (i)

Section 2's maps are all at one level (`DEPTH_M`). This section shows
**temperature/salinity/current at four depth levels at once** -- surface, 120 m, 300 m,
1000 m -- one row per depth, columns (CROCO, Copernicus, bias). 
Useful for spotting depth-dependent biases (e.g. a warm surface bias that reverses sign at
depth) that a single-level map can't show.


In [61]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    depth_salt_figs = {}
    for day in CYCLE_DAYS:
        depth_salt_figs[day] = val.compare_forecast_depth_levels(
            CROCO_HIS, REFERENCE, var='salt', date=day,
            depths=(0, 120, 300, 1000), Yorig=YORIG, daily_mean=True,
            out=os.path.join(VALIDATION_DIR, f'depth_levels_salt_vs_forecast_{day}.png'))
    print(f"-> {len(depth_salt_figs)} figure(s) written, one per day: {CYCLE_DAYS}")
else:
    print("Copernicus Marine Salinity Forecast unavailable or REFERENCE missing - skipping this comparison.")

-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


In [62]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    depth_salt_figs = {}
    for day in CYCLE_DAYS:
        depth_salt_figs[day] = val.compare_forecast_depth_levels(
            CROCO_HIS, REFERENCE, var='temp', date=day,
            depths=(0, 120, 300, 1000), Yorig=YORIG, daily_mean=True,
            out=os.path.join(VALIDATION_DIR, f'depth_levels_temp_vs_forecast_{day}.png'))
    print(f"-> {len(depth_salt_figs)} figure(s) written, one per day: {CYCLE_DAYS}")
else:
    print("Copernicus Marine Salinity Forecast unavailable or REFERENCE missing - skipping this comparison.")

-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


In [63]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    depth_salt_figs = {}
    for day in CYCLE_DAYS:
        depth_salt_figs[day] = val.compare_forecast_depth_levels(
            CROCO_HIS, REFERENCE, var='speed', date=day,
            depths=(0, 120, 300, 1000), Yorig=YORIG, daily_mean=True,
            out=os.path.join(VALIDATION_DIR, f'depth_levels_speed_vs_forecast_{day}.png'))
    print(f"-> {len(depth_salt_figs)} figure(s) written, one per day: {CYCLE_DAYS}")
else:
    print("Copernicus Marine Salinity Forecast unavailable or REFERENCE missing - skipping this comparison.")

-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


## 3. Scatter plots -- pointwise CROCO vs Copernicus Marine Forecast

The bias maps above show *where* the differences are; a scatter plot of
every grid point (CROCO value vs. co-located reference value) shows the
overall *shape* of the agreement -- a tight cloud along the 1:1 line means
good agreement; a cloud offset from the line indicates a systematic bias;
a fan-shaped cloud indicates the CROCO field is over/under-dispersed
relative to the reference (this is exactly what the Taylor diagram below
summarises in one number: the model/reference standard-deviation ratio).


In [64]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    scatter_figs = {}
    for day in CYCLE_DAYS:
        ti = val._tindex_for_date(ds, day, -1)   # CROCO record matching this day
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        ssh = ds['zeta'].isel(time=ti).values
        plon, plat, pfield = val.load_parent(REFERENCE, 'ssh', date=day)
        ssh_ref = val.regrid_to_croco(plon, plat, pfield, ds)
        val.scatter_vs_reference(ssh, ssh_ref, 'SSH', 'm', ax=axes[0])
        
        sss = pp.surface(ds, 'salt', tindex=ti).values
        plon, plat, pfield = val.load_parent(REFERENCE, 'salt', date=day)
        sss_ref = val.regrid_to_croco(plon, plat, pfield, ds)
        val.scatter_vs_reference(sss, sss_ref, 'SSS', 'PSU', ax=axes[1])
        
        sst = pp.surface(ds, 'temp', tindex=ti).values
        plon, plat, pfield = val.load_parent(REFERENCE, 'temp', date=day)
        sst_ref = val.regrid_to_croco(plon, plat, pfield, ds)
        val.scatter_vs_reference(sst, sst_ref, 'SST', 'degC', ax=axes[2])

        fig.suptitle(day)
        fig.tight_layout()
        out_png = os.path.join(VALIDATION_DIR, f"scatter_sss_sst_ssh_vs_forecast_{day}.png")
        fig.savefig(out_png, dpi=250, bbox_inches="tight")
        plt.close(fig)
        scatter_figs[day] = out_png
    print(f"-> {len(scatter_figs)} figure(s) written, one per day: {list(scatter_figs)}")
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")
    

-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


## 4. Taylor diagram

A **Taylor diagram** summarises three skill statistics in a single polar
plot: the correlation with the reference (angle), the ratio of the model's
standard deviation to the reference's (radial distance), and -- implicitly,
via distance from the reference point -- the centred RMSE. It's the
standard GODAE OceanView / CMEMS intercomparison summary plot, and is what
`sftools.validation_godae` (module `vg` here) was built to produce; see
that module's docstring for the full GODAE metric set (bias, RMSD, unbiased
RMSD, correlation, two scatter-index variants, std-ratio).

We score SST, SSH, SSS and surface current speed against the reference in
one pass and put them all on the same diagram, so a single glance shows
which variable(s) are driving any V1 concern.


In [65]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    # All four variables now go through the same call -- SSH's arbitrary-
    # geoid-reference issue (CROCO zeta has no absolute reference level, so a
    # raw-level comparison would score that mismatch as spurious bias) is
    # handled inside godae_scorecard_croco_vs_glorys itself now (ssh_anomaly=True
    # by default), so no special-casing is needed here any more.
    #
    # Scored ONE DAY AT A TIME (rather than just the cycle's last time step),
    # so the Taylor diagram below can plot one figure per day too.
    rows = []
    report_by_day = {}
    for day in CYCLE_DAYS:
        day_rows = []
        for var in ('temp', 'ssh', 'salt', 'speed'):
            s = val.godae_scorecard_croco_vs_glorys(CROCO_HIS, REFERENCE, var, date=day, Yorig=YORIG)
            day_rows.append({**s, 'vs': 'reference', 'layer': 'all'})
        report_by_day[day] = pd.DataFrame(day_rows)
        rows.extend(day_rows)
        print(f"-- {day} --")
        val.print_scorecard_table(report_by_day[day])

    # every day x every variable; Section 6's pass/fail summary below uses
    # the LAST day (rows was built day-by-day, so by_var there naturally
    # ends up holding that last day's scores).
    report = pd.DataFrame(rows)
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping GODAE scorecard (Sections 4/6 below will be skipped too).")
    report = pd.DataFrame(columns=['variable','bias','rmsd','urmsd','corr','std_ratio','vs','layer'])
    report_by_day = {}
    

-- 2026-07-11 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209 -0.160 0.329  0.287 0.991  1.182  13.805
     ssh reference   all 8209 -0.001 0.009  0.009 0.987 22.161  15.810
    salt reference   all 8209  0.008 0.071  0.070 0.979  0.195  20.823
   speed reference   all 8209 -0.006 0.101  0.101 0.641 54.754  86.028
-- 2026-07-12 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209 -0.272 0.443  0.350 0.987  1.433  16.666
     ssh reference   all 8209 -0.001 0.010  0.010 0.985 24.163  17.408
    salt reference   all 8209  0.011 0.091  0.090 0.965  0.249  26.825
   speed reference   all 8209  0.012 0.084  0.083 0.663 53.084  84.348
-- 2026-07-13 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209 -0.240 0.435  0.363 0.985  1.483  17.764
     ssh reference   all 8209 -0.001 0.011  0.011 0.982 25.879  18.869
    salt reference   all 8

In [66]:
if not report.empty:
    taylor_figs = {}
    for day, rep in report_by_day.items():
        fig = plt.figure(figsize=(7, 7))
        ax = fig.add_subplot(111, polar=True)
        has_neg = bool((rep['corr'] < 0).any())
        thetamax = 180 if has_neg else 90
        corr_ticks = ([-1.0, -0.5, 0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0] if has_neg
                      else [0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0])
        ax.set_thetamin(0); ax.set_thetamax(thetamax)
        ax.set_xticks(np.arccos(corr_ticks)); ax.set_xticklabels([str(c) for c in corr_ticks])
        ax.set_rlabel_position(0)   # keep the std-dev tick labels along the bottom (theta=0) axis
        finite = rep['std_ratio'][np.isfinite(rep['std_ratio'])]
        r_max = max(1.6, finite.max() * 1.2) if len(finite) else 1.6
        ax.set_ylim(0, r_max)
        ax.plot(0, 1, 'k*', ms=16, label='reference')
        for _, row in rep.iterrows():
            theta = np.arccos(np.clip(row['corr'], -1, 1))
            ax.plot(theta, row['std_ratio'], 'o', ms=10,
                   label=f"{row['variable']}  (RMSD={row['rmsd']:.2f})")

        # bottom-axis title: placed in AXES-FRACTION coords (not polar data
        # coords), centered under the whole plot and below the std-dev tick
        # numbers -- immune to the tick labels' own placement/length.
        ax.text(0.5, -0.08, 'Normalised standard deviation (CROCO / reference)',
               transform=ax.transAxes, ha='center', va='top', fontsize=10)
        # arc-axis title -- correlation, curving along the outer arc at
        # mid-angle, rotated tangent to the arc so it follows its curvature.
        ax.text(np.radians(thetamax / 6), r_max * 1.25, 'Correlation coefficient',
               ha='center', va='center', fontsize=10,
               rotation=90 - thetamax / 2, rotation_mode='anchor')

        ax.set_title(f'Taylor diagram -- CROCO vs reference ({day})', pad=20)
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=9)
        fig.tight_layout()
        out_png = os.path.join(VALIDATION_DIR, f"taylor_diagram_{day}.png")
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        taylor_figs[day] = out_png
    print(f"-> {len(taylor_figs)} figure(s) written, one per day: {list(taylor_figs)}")
else:
    print("No GODAE scorecard available (Copernicus Marine Forecast unavailable) - skipping Taylor diagram.")
    

-> 6 figure(s) written, one per day: ['2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']


## 5. Automated pass/fail summary (V1)
# CROCO Forecast vs Copernicus Marine Forecast

Pulling together every criterion from the Forecasting Accuracy Validation
Criteria table (Technical Specification Section 9.3) into one summary.

Each criterion is scored against the **cycle mean** of the per-day GODAE
scorecard (Section 4, `report`, one row per day per variable) -- rather
than a single time step, so one unusually good or bad day can't flip the
pass/fail on its own. The first **`SPINUP_DAYS`** day(s) of the forecast
are excluded from this mean: CROCO's first day(s) after cold/warm-starting
a cycle carry spin-up transients (adjustment to the boundary/initial
conditions) that are a known model-startup artefact, not a genuine skill
problem -- scoring them would unfairly bias the cycle-mean criteria below.
Sections 2--5c still show every day, spin-up included, for inspection.

The worst single day (among the non-spin-up days) for each headline
variable is also reported for context; see Section 5b (CROCO vs parent)
and 5c (CROCO vs satellite) for the full day-by-day, domain-wide spread
behind these cycle-mean numbers.


In [67]:
# Forecast spin-up: the first SPINUP_DAYS day(s) of a cycle carry model
# start-up transients (adjustment to IC/BC) -- excluded from the cycle-mean
# pass/fail criteria below (Sections 2-5c still show every day, unfiltered).
SPINUP_DAYS = 2

if not report.empty:
    if len(CYCLE_DAYS) > SPINUP_DAYS:
        EVAL_DAYS = CYCLE_DAYS[SPINUP_DAYS:]
    else:
        EVAL_DAYS = CYCLE_DAYS[:]
        print(f"Cycle only has {len(CYCLE_DAYS)} day(s) -- shorter than SPINUP_DAYS="
             f"{SPINUP_DAYS}, so no day can be excluded; using every day.")
    report_eval = report[report['date'].isin(EVAL_DAYS)]
    
    by_var = (report_eval.groupby('variable')[['bias', 'rmsd', 'urmsd', 'corr', 'si', 'si_std', 'std_ratio']]
              .mean().to_dict('index'))

    criteria = [
        ('SST cycle-mean domain-avg RMSD < 0.5 degC',       by_var['temp']['rmsd'] < 0.5,  f"{by_var['temp']['rmsd']:.3f} degC"),
        ('SSH cycle-mean spatial correlation > 0.90',       by_var['ssh']['corr'] > 0.90, f"{by_var['ssh']['corr']:.3f}"),
        ('Salinity cycle-mean domain-avg |bias| < 0.2 PSU', abs(by_var['salt']['bias']) < 0.2, f"{by_var['salt']['bias']:+.3f} PSU"),
        ('Numerical stability (no NaN/Inf)',                stability_ok, 'see section 1'),
    ]

    print(f"Cycle-mean scorecard: {len(EVAL_DAYS)} of {len(CYCLE_DAYS)} day(s) "
         f"(excluding {CYCLE_DAYS[:SPINUP_DAYS] if len(CYCLE_DAYS) > SPINUP_DAYS else []} as spin-up): {EVAL_DAYS}")
    print()
    print(f"{'Criterion':45s} {'Result':6s}  Value")
    print('-' * 70)
    all_pass = True
    for name, passed, value in criteria:
        all_pass &= passed
        print(f"{name:45s} {'PASS' if passed else 'FAIL':6s}  {value}")
    print('-' * 70)
    
    # Worst single day (among EVAL_DAYS) per headline variable -- a cycle
    # mean can mask one bad day; this flags it, and Section 5b's boxplot
    # shows the full spread.
    if len(EVAL_DAYS) > 1:
        temp_rows = report_eval[report_eval['variable'] == 'temp']
        ssh_rows  = report_eval[report_eval['variable'] == 'ssh']
        salt_rows = report_eval[report_eval['variable'] == 'salt']
        worst_temp = temp_rows.loc[temp_rows['rmsd'].idxmax()]
        worst_ssh  = ssh_rows.loc[ssh_rows['corr'].idxmin()]
        worst_salt = salt_rows.loc[salt_rows['bias'].abs().idxmax()]
        print()
        print(f"Worst single day -- SST RMSD:   {worst_temp['date']}  ({worst_temp['rmsd']:.3f} degC)")
        print(f"Worst single day -- SSH corr:   {worst_ssh['date']}  ({worst_ssh['corr']:.3f})")
        print(f"Worst single day -- Salt bias:  {worst_salt['date']}  ({worst_salt['bias']:+.3f} PSU)")
    print()
    ds.close()
else:
    print("Overall V1 status: SKIPPED -- Copernicus Marine Forecast unavailable, "
         "no GODAE scorecard to build the pass/fail summary from.")
    ds.close()
    

Cycle-mean scorecard: 4 of 6 day(s) (excluding ['2026-07-11', '2026-07-12'] as spin-up): ['2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16']

Criterion                                     Result  Value
----------------------------------------------------------------------
SST cycle-mean domain-avg RMSD < 0.5 degC     FAIL    0.579 degC
SSH cycle-mean spatial correlation > 0.90     PASS    0.976
Salinity cycle-mean domain-avg |bias| < 0.2 PSU PASS    +0.019 PSU
Numerical stability (no NaN/Inf)              PASS    see section 1
----------------------------------------------------------------------

Worst single day -- SST RMSD:   2026-07-16  (0.761 degC)
Worst single day -- SSH corr:   2026-07-16  (0.966)
Worst single day -- Salt bias:  2026-07-16  (+0.023 PSU)



## 6. Time series comparison

A single-point time series makes the *temporal* behaviour visible in a way
maps can't -- useful for checking that CROCO isn't drifting away from the
reference over the run (a common failure mode in a hindcast with weak/no
nudging). Picked here at a coastal point, since that's also where the
upwelling exercise in `04_exercises.ipynb` focuses.

CROCO's (usually sub-daily) output is first collapsed to one **daily
mean** per calendar day, matching the daily cadence of the Mercator
Forecast and satellite reference products (Section 1c) -- comparing a
sub-daily CROCO record against a daily-mean reference would otherwise
alias tidal/diurnal CROCO variability into spurious disagreement.

Temperature and salinity are taken at **`DEPTH_M`** (set in Section 0 --
`None` -> surface, else that true depth), the same setting Section 2's
grid comparisons use. SSH has no depth dimension, so it's always surface
regardless of `DEPTH_M` (as in Section 2). Satellite SST/SSS are
surface-only products by construction, so they're only plotted when
`DEPTH_M is None`; at a non-surface `DEPTH_M` those two lines are skipped
and only CROCO-vs-parent is shown for temperature/salinity.

One figure, three stacked subpanels sharing a time axis:

- **SSH** (top): CROCO (daily mean) vs. the Copernicus Marine Forecast parent.
- **Temperature**: CROCO (daily mean) vs. parent, plus OSTIA and ODYSSEA satellite SST if `DEPTH_M is None`.
- **Salinity**: CROCO (daily mean) vs. parent, plus SMOS satellite SSS if `DEPTH_M is None`.


In [68]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    clon, clat, cmask = pp.lonlatmask(ds)

    if not PROFILE_POINTS:
        # a coastal-ish point: 75% of the way across the grid, mid-latitude
        j0 = ds.sizes['eta_rho'] // 2
        i0 = int(0.75 * ds.sizes['xi_rho'])
        lon0, lat0 = float(clon[j0, i0]), float(clat[j0, i0])
    else: # you define your profile point ==> take the first (or that one you want) lon-lat point
        j0 = PROFILE_POINTS[0,0]
        i0 = PROFILE_POINTS[1,0]
        lon0, lat0 = float(clon[j0, i0]), float(clat[j0, i0])
    
    dlab = 'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'
    print(f'time series point: ({lon0:.2f}, {lat0:.2f})  --  temp/salt at {dlab}')

    # ---- CROCO: collapse the (usually sub-daily) history output to one
    # daily mean per calendar day, matching the daily cadence of the parent
    # and satellite products below (date=None -> average EVERY day, not just
    # one). temp/salt at DEPTH_M (Section 0, None -> surface); SSH has no
    # depth dimension so it always stays at the surface, same as Section 2.
    ds_daily = val._maybe_daily_mean(ds, date=None, daily_mean=True)
    temp_croco = pp.timeseries(ds_daily, 'temp', lon0, lat0, depth_m=DEPTH_M)
    salt_croco = pp.timeseries(ds_daily, 'salt', lon0, lat0, depth_m=DEPTH_M)
    ssh_croco  = pp.timeseries(ds_daily, 'zeta', lon0, lat0)
    speed_croco = pp.timeseries(ds_daily, 'speed', lon0, lat0, depth_m=DEPTH_M)
    ct = temp_croco['time'].values

    # ---- Copernicus Marine Forecast (parent), same point and same DEPTH_M
    # for temp/salt (zos/SSH has no depth axis, always surface). ----
    dsp = xr.open_dataset(REFERENCE)
    def _parent_point_series(cmems_var, depth_m=None):
        da = dsp[cmems_var]
        if "depth" in da.dims:
            da = da.isel(depth=0) if depth_m is None else da.sel(depth=abs(depth_m), method="nearest")
        da = da.sel(longitude=lon0, latitude=lat0, method="nearest")
        return da['time'].values, np.atleast_1d(da.values)
    pt_temp, pv_temp = _parent_point_series('thetao', depth_m=DEPTH_M)
    pt_salt, pv_salt = _parent_point_series('so', depth_m=DEPTH_M)
    pt_ssh,  pv_ssh  = _parent_point_series('zos')
    pt_u, pv_u = _parent_point_series('uo', depth_m=DEPTH_M)
    pt_v, pv_v = _parent_point_series('vo', depth_m=DEPTH_M)
    pv_speed = np.sqrt(np.asarray(pv_u) ** 2 + np.asarray(pv_v) ** 2)
    dsp.close()

    # ---- Satellite SST/SSS: surface-only products by construction, so only
    # plotted when DEPTH_M is None (each day's file is already a flat
    # (lon, lat) grid -- no depth axis to select from either way). ----
    def _satellite_point_series(product, avail_key):
        if DEPTH_M is not None:
            return [], []
        files = SAT_FILES.get(product, {})
        if not AVAIL.get(avail_key, False) or not files:
            return [], []
        days_, vals_ = [], []
        for day, fname in sorted(files.items()):
            if not fname or not os.path.exists(fname):
                continue
            loaded = val.load_satellite_field(fname, product, date=day)
            if loaded is None:
                continue
            plon, plat, pfield = loaded
            # some products (e.g. SMOS) can come back with a leftover
            # length-1 leading dim (time/other) that load_satellite_field
            # didn't squeeze out -- align it with the 2D (lon, lat) grid
            # before indexing, or skip this day if it still doesn't match.
            pfield = np.squeeze(np.asarray(pfield))
            if pfield.shape != plon.shape:
                print(f"  [{product}] {day}: field shape {pfield.shape} doesn't match "
                     f"grid shape {plon.shape} - skipping")
                continue
            d2 = (plon - lon0) ** 2 + (plat - lat0) ** 2
            j, i = np.unravel_index(np.argmin(d2), d2.shape)
            v = pfield[j, i]
            if np.isfinite(v):
                days_.append(np.datetime64(day)); vals_.append(v)
        return days_, vals_

    ostia_t, ostia_v = _satellite_point_series('OSTIA', 'ostia_l4')
    odyssea_t, odyssea_v = _satellite_point_series('ODYSSEA', 'odyssea_l3s')
    smos_t, smos_v = _satellite_point_series('SMOS', 'smos_l4_sss')
    if DEPTH_M is not None:
        print(f"DEPTH_M={DEPTH_M:g} m set -- satellite SST/SSS are surface-only products, "
             "so OSTIA/ODYSSEA/SMOS are omitted below (parent/CROCO panels still shown).")

    # ---- one figure, 4 stacked subpanels: SSH (top), temp, salt, currents ----
    fig, axes = plt.subplots(4, 1, figsize=(9, 13), sharex=True)

    axes[0].plot(ct, ssh_croco.values, 'o-', color='C0', ms=3, lw=1.4, label='CROCO)')
    axes[0].plot(pt_ssh, pv_ssh, 's--', color='C3', ms=4, lw=1.4, label='parent')
    axes[0].set_ylabel('SSH (m)')
    axes[0].set_title(f'SSH')
    axes[0].legend(framealpha=0.5, fontsize=10)

    axes[1].plot(ct, temp_croco.values, 'o-', color='C0', ms=3, lw=1.4, label='CROCO')
    axes[1].plot(pt_temp, pv_temp, 's--', color='C3', ms=4, lw=1.4, label='parent')
    if ostia_t:
        axes[1].plot(ostia_t, ostia_v, '^:', color='C1', ms=5, lw=1.2, label='OSTIA')
    if odyssea_t:
        axes[1].plot(odyssea_t, odyssea_v, 'v:', color='C2', ms=5, lw=1.2, label='ODYSSEA')
    axes[1].set_ylabel('temperature (degC)')
    axes[1].set_title(f'temperature  ({dlab})')
    axes[2].legend(framealpha=0.5, fontsize=10)

    axes[2].plot(ct, salt_croco.values, 'o-', color='C0', ms=3, lw=1.4, label='CROCO')
    axes[2].plot(pt_salt, pv_salt, 's--', color='C3', ms=4, lw=1.4, label='parent')
    if smos_t:
        axes[2].plot(smos_t, smos_v, '^:', color='C4', ms=5, lw=1.2, label='SMOS')
    axes[2].set_ylabel('salinity (PSU)')
    axes[2].set_title(f'salinity  ({dlab})')
    axes[2].legend(framealpha=0.5, fontsize=10)

    axes[3].plot(ct, speed_croco.values, 'o-', color='C0', ms=3, lw=1.4, label='CROCO')
    axes[3].plot(pt_u, pv_speed, 's--', color='C3', ms=4, lw=1.4, label='parent')
    axes[3].set_ylabel('speed (m s$^{-1}$)')
    axes[3].set_title(f'current speed  ({dlab})')
    axes[3].set_xlabel('time')
    axes[3].legend(framealpha=0.5, fontsize=10)

    plt.suptitle(f"Location: ({lon0:.2f}, {lat0:.2f}) \n")
    for ax in axes:
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(os.path.join(VALIDATION_DIR, f'timeseries_vs_forecast_{lon0:.2f}E_{lat0:.2f}N.png'), dpi=300, bbox_inches='tight')
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")

time series point: (-17.11, 18.99)  --  temp/salt at surface


### 6b. Domain-wide bias boxplot (per day)

Section 5 tracks a single point over time; this subsection instead shows
the *spread* of the CROCO-minus-parent difference across the WHOLE domain,
one box per day, for the same variables (SSH, temperature, salinity and current)
-- temperature/salinity/current at `DEPTH_M` (Section 0, same as Sections 2 and 5),
SSH always at the surface (no depth axis) and compared as an anomaly
(domain mean removed from both sides), exactly as Section 2's
`compare_ssh` does.

A wide box / long whiskers on a given day flags spatially inconsistent
bias that a single-point time series (Section 5) could miss entirely; a
box drifting away from zero across the cycle flags a growing systematic
bias. Complements, rather than replaces, the per-day bias maps in Section 2.


In [69]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    dlab = 'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'

    # data computation now lives in sftools.validation.domain_diff (shared
    # with Section 6c's satellite version below, and with anywhere else
    # that needs a domain-wide CROCO-vs-reference difference for one day)
    ssh_diffs   = [val.domain_diff(ds, REFERENCE, 'ssh',   day) for day in CYCLE_DAYS]
    temp_diffs  = [val.domain_diff(ds, REFERENCE, 'temp',  day, depth_m=DEPTH_M) for day in CYCLE_DAYS]
    salt_diffs  = [val.domain_diff(ds, REFERENCE, 'salt',  day, depth_m=DEPTH_M) for day in CYCLE_DAYS]
    speed_diffs = [val.domain_diff(ds, REFERENCE, 'speed', day, depth_m=DEPTH_M) for day in CYCLE_DAYS]

    def _stacked_boxplot(diffs_list, ylabels, titles, out, metric='bias'):
        data = [[np.abs(d) for d in diffs] for diffs in diffs_list] if metric == 'rmse' else diffs_list
        fig, axes = plt.subplots(len(data), 1, figsize=(max(6, 1.1 * len(CYCLE_DAYS)), 3.3 * len(data)),
                                 sharex=True)
        positions = np.arange(1, len(CYCLE_DAYS) + 1)
        for ax, diffs, ylabel, title in zip(axes, data, ylabels, titles):
            ax.boxplot(diffs, positions=positions, showfliers=False)
            ax.set_xticks(positions); ax.set_xticklabels(CYCLE_DAYS)
            if metric == 'bias':
                ax.axhline(0, color='k', ls='--', lw=1)
            ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(alpha=0.3)
        axes[-1].set_xlabel('day')
        plt.setp(axes[-1].get_xticklabels(), rotation=45, ha='right')
        fig.suptitle("CROCO - parent" if metric == 'bias' else "|CROCO - parent|  (RMSE-spread per day)")
        fig.tight_layout()
        fig.savefig(out, dpi=150, bbox_inches='tight')
        return fig

    diffs_all = [ssh_diffs, temp_diffs, salt_diffs, speed_diffs]
    ylabels = ["SSH' bias (m)", 'temperature bias (degC)', 'salinity bias (PSU)', 'speed bias (m s$^{-1}$)']
    titles = ['SSH anomaly bias', f'temperature bias  ({dlab})', f'salinity bias  ({dlab})',
             f'current speed bias  ({dlab})']

    _stacked_boxplot(diffs_all, ylabels, titles,
                     os.path.join(VALIDATION_DIR, 'boxplot_bias_vs_forecast.png'), metric='bias')

    rmse_ylabels = [y.replace('bias', '|error|') for y in ylabels]
    rmse_titles = [t.replace('bias', 'RMSE-spread') for t in titles]
    _stacked_boxplot(diffs_all, rmse_ylabels, rmse_titles,
                     os.path.join(VALIDATION_DIR, 'boxplot_rmse_vs_forecast.png'), metric='rmse')
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")


## – Full‑domain time series with spatial spread (CROCO vs parent)

In [70]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    dlab = 'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'

    def _domain_mean_std(var, depth_m=None):
        m_c, s_c, m_p, s_p = [], [], [], []
        for day in CYCLE_DAYS:
            croco2d, parent2d = val._field_pair_for_day(
                ds, REFERENCE, var, day, depth_m, tindex=-1, daily_mean=True, margin_deg=None)
            m_c.append(np.nanmean(croco2d)); s_c.append(np.nanstd(croco2d))
            m_p.append(np.nanmean(parent2d)); s_p.append(np.nanstd(parent2d))
        return map(np.array, (m_c, s_c, m_p, s_p))

    specs = [('ssh', "SSH' (m)", 'SSH anomaly', None),
            ('temp', 'temperature (degC)', f'temperature ({dlab})', DEPTH_M),
            ('salt', 'salinity (PSU)', f'salinity ({dlab})', DEPTH_M),
            ('speed', 'speed (m s$^{-1}$)', f'current speed ({dlab})', DEPTH_M)]

    fig, axes = plt.subplots(len(specs), 1, figsize=(max(6, 1.1 * len(CYCLE_DAYS)), 3.3 * len(specs)),
                             sharex=True)
    x = np.arange(1, len(CYCLE_DAYS) + 1)
    for ax, (var, ylabel, title, dm) in zip(axes, specs):
        m_c, s_c, m_p, s_p = _domain_mean_std(var, depth_m=dm)
        ax.plot(x, m_c, 'o-', color='C0', lw=1.5, ms=5, label='CROCO')
        ax.fill_between(x, m_c - s_c, m_c + s_c, color='C0', alpha=0.2)
        ax.plot(x, m_p, 's--', color='C3', lw=1.5, ms=5, label='parent')
        ax.fill_between(x, m_p - s_p, m_p + s_p, color='C3', alpha=0.2)
        ax.set_xticks(x); ax.set_xticklabels(CYCLE_DAYS)
        ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    axes[-1].set_xlabel('day')
    plt.setp(axes[-1].get_xticklabels(), rotation=45, ha='right')
    fig.suptitle("CROCO vs parent -- full-domain mean +/- spatial std")
    fig.tight_layout()
    fig.savefig(os.path.join(VALIDATION_DIR, 'timeseries_mean_domain_vs_forecast.png'), dpi=150, bbox_inches='tight')
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")
    

### 6c. CROCO vs satellite domain-wide bias boxplot (per day)

Same idea as Section 5b, but against the independent satellite products
instead of the Copernicus Marine Forecast parent -- OSTIA and ODYSSEA
(SST, "class 3" in the GODAE OceanView taxonomy, see Section 7) grouped
side by side per day, and SMOS (SSS, Section 7b) on its own panel. Uses
the same regridding as `sftools.validation_satellite.compare_satellite_grid`
(Section 7/7b), just showing the domain-wide spread of the bias as boxes
instead of one figure per day per product.

Satellite SST/SSS are surface-only products, so -- same rule as Sections 5
and 5b -- this subsection only runs when `DEPTH_M is None`; a product with
no downloaded/available files for this cycle is skipped (not failed),
consistent with Sections 7/7b.


In [71]:
sat_avail_key = {"OSTIA": "ostia_l4", "ODYSSEA": "odyssea_l3s", "SMOS": "smos_l4_sss"}
any_sat_avail = any(AVAIL.get(sat_avail_key[p], False) and SAT_FILES.get(p) for p in sat_avail_key)

if not any_sat_avail:
    print("No satellite product (OSTIA/ODYSSEA/SMOS) available/downloaded for this cycle - skipping.")
else:
    def _sat_diff_for_day(product, day):
        '''CROCO-minus-satellite difference for one product/day, built the
        same way Sections 7/7b regrid satellite onto the CROCO grid --
        val.domain_diff_satellite() just flattens/filters/subsamples the
        (croco, satellite) pair once it's been loaded+regridded here.'''
        files = SAT_FILES.get(product, {})
        fname = files.get(day)
        if not AVAIL.get(sat_avail_key[product], False) or not fname or not os.path.exists(fname):
            return np.array([])
        loaded = val.load_satellite_field(fname, product, date=day)
        if loaded is None:
            return np.array([])
        plon, plat, pfield = loaded
        pfield = np.squeeze(np.asarray(pfield))       # guard the same shape quirk as Section 5
        if pfield.shape != plon.shape:
            return np.array([])
        try:
            clon, clat, croco_field = val._croco_field_satellite(CROCO_HIS, product, day, Yorig=YORIG)
        except Exception:
            return np.array([])
        grid = xr.Dataset(
            {"mask_rho": (("eta_rho", "xi_rho"), np.where(np.isfinite(croco_field), 1.0, np.nan))},
            coords={"lon_rho": (("eta_rho", "xi_rho"), clon), "lat_rho": (("eta_rho", "xi_rho"), clat)})
        sat_on_croco = val.regrid_to_croco(plon, plat, pfield, grid)
        return val.domain_diff_satellite(croco_field, sat_on_croco)

    ostia_diffs   = [_sat_diff_for_day('OSTIA',   day) for day in CYCLE_DAYS]
    odyssea_diffs = [_sat_diff_for_day('ODYSSEA', day) for day in CYCLE_DAYS]
    smos_diffs    = [_sat_diff_for_day('SMOS',    day) for day in CYCLE_DAYS]

    sst_groups = {"OSTIA": ostia_diffs, "ODYSSEA": odyssea_diffs}
    sss_groups = {"SMOS": smos_diffs}

    ## bias
    val.bias_boxplot_multi(sst_groups, CYCLE_DAYS, 'temperature bias (degC)',
                           'CROCO - satellite SST bias', colors=['C1', 'C2'],
                           out=os.path.join(VALIDATION_DIR, 'boxplot_bias_sst_vs_satellite.png'))
    val.bias_boxplot_multi(sss_groups, CYCLE_DAYS, 'salinity bias (PSU)',
                           'CROCO - satellite SSS bias', colors=['C4'],
                           out=os.path.join(VALIDATION_DIR, 'boxplot_bias_sss_vs_satellite.png'))
    ## RMSE
    val.bias_boxplot_multi(sst_groups, CYCLE_DAYS, 'temperature |error| (degC)',
                           'CROCO - satellite SST RMSE-spread', colors=['C1', 'C2'], metric='rmse',
                           out=os.path.join(VALIDATION_DIR, 'boxplot_rmse_sst_vs_satellite.png'))
    val.bias_boxplot_multi(sss_groups, CYCLE_DAYS, 'salinity |error| (PSU)',
                           'CROCO - satellite SSS RMSE-spread', colors=['C4'], metric='rmse',
                           out=os.path.join(VALIDATION_DIR, 'boxplot_rmse_sss_vs_satellite.png'))


## 7. Satellite SST validation map -- OSTIA & ODYSSEA (ii)

CROCO forecast SST against two independent Copernicus Marine satellite SST
products, each availability-guarded and skipped (not failed) if the
product or a given day's file isn't reachable:

- **OSTIA** (`SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001`, subdataset
  `METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2`) -- Level 4, multi-sensor,
  gap-filled analysis, 0.05 deg daily. The "best available" gridded SST.
- **ODYSSEA** (`SST_GLO_SST_L3S_NRT_OBSERVATIONS_010_010`, subdataset
  `IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE`) -- Level 3S, single-
  sensor-type composite with real swath/cloud gaps -- closer to the raw
  satellite retrieval, no gap-filling, so it's a useful independent check
  against OSTIA's L4 interpolation.

For each product: a **separate figure per day** in the forecast cycle
(3 panels: CROCO SST | satellite SST | bias map), via
`sftools.validation_satellite.compare_satellite_grid`, plus per-day
bias/RMSE/centred-RMSE/correlation (same `domain_statistics` engine as
Section 2). SMOS SSS follows the same pattern, in Section 7b below.

**Class of comparison:** this is "class 3" in the GODAE OceanView taxonomy
(model vs. independent satellite retrieval, as opposed to class 1/2 model-
vs-reanalysis in Section 2 or class 4 model-vs-in-situ in Section 8) --
satellite SST has its own retrieval uncertainty (skin-vs-bulk temperature
difference, residual cloud contamination for L3S) so a satellite bias
should be read alongside, not instead of, the GLORYS/Mercator bias above.


In [72]:
# SAT_FILES was built in Section 1c: {"OSTIA": {date: path, ...}, "ODYSSEA": {...}}
sat_avail_key = {"OSTIA": "ostia_l4", "ODYSSEA": "odyssea_l3s"}
sat_stats_all = {}
for product, files in SAT_FILES.items():
    if product == "SMOS":
        continue   # SMOS SSS is handled separately in Section 7b below
    if not AVAIL[sat_avail_key[product]]:
        print(f"{product} unavailable on the CMEMS platform - skipping.")
        continue
    if not files:
        print(f"{product}: nothing downloaded for this cycle (no files/coverage) - skipping.")
        continue
    figs, stats = val.compare_satellite_grid(CROCO_HIS, files, product, Yorig=YORIG,
                                            out_dir=VALIDATION_DIR)
    sat_stats_all[product] = stats
    print(f"  -> {len(figs)} figure(s) written, one per day: "
         f"{[os.path.basename(p) for p in figs.values()]}")
    if stats is not None and not stats.empty:
        print(f"\n{product} daily SST statistics:")
        print(stats.to_string(index=False))
        stats.to_csv(os.path.join(VALIDATION_DIR, f"sst_vs_{product.lower()}_stats.csv"), index=False)

print()


  [OSTIA] 2026-07-11:
  [SST]  n=8257  bias=+0.146  RMSE=0.737  cRMSE=0.723  corr=0.934
  [OSTIA] 2026-07-12:
  [SST]  n=8257  bias=+0.058  RMSE=0.749  cRMSE=0.747  corr=0.928
  [OSTIA] 2026-07-13:
  [SST]  n=8257  bias=+0.102  RMSE=0.705  cRMSE=0.697  corr=0.933
  [OSTIA] 2026-07-14:
  [SST]  n=8257  bias=+0.032  RMSE=0.662  cRMSE=0.662  corr=0.948
  [OSTIA] 2026-07-15:
  [SST]  n=8257  bias=+0.047  RMSE=0.667  cRMSE=0.665  corr=0.945
  [OSTIA] 2026-07-16:
  [SST]  n=8257  bias=-0.120  RMSE=0.803  cRMSE=0.794  corr=0.944
  -> 6 figure(s) written, one per day: ['sst_vs_ostia_2026-07-11.png', 'sst_vs_ostia_2026-07-12.png', 'sst_vs_ostia_2026-07-13.png', 'sst_vs_ostia_2026-07-14.png', 'sst_vs_ostia_2026-07-15.png', 'sst_vs_ostia_2026-07-16.png']

OSTIA daily SST statistics:
     n      bias     rmse    crmse     corr  model_mean  ref_mean  model_min  model_max   ref_min   ref_max       date
8257.0  0.146031 0.737426 0.722822 0.933759   24.150640 24.004609   4.771338  28.806669 17.932797 

## 7b. Satellite SSS validation -- SMOS (ii)

CROCO forecast sea surface salinity against the Copernicus Marine SMOS
Level-4 SSS product -- same availability-guarded, skip-not-fail design as
the OSTIA/ODYSSEA SST comparisons above, via
`sftools.validation_satellite.compare_satellite_grid` (product="SMOS"),
one figure per day.

**Note:** the SMOS dataset id in `sftools.download.cmems.VALIDATION_DATASETS`
(`smos_l4_sss`) has not been confirmed against the live CMEMS catalogue --
verify it with `copernicusmarine.describe` before relying on this section
in production; until then it's expected to report "unavailable" and skip
cleanly rather than fail.


In [73]:
smos_files = SAT_FILES.get("SMOS", {})
if not AVAIL['smos_l4_sss']:
    print("SMOS SSS unavailable on the CMEMS platform - skipping.")
elif not smos_files:
    print("SMOS: nothing downloaded for this cycle (no files/coverage) - skipping.")
else:
    figs, smos_stats = val.compare_satellite_grid(CROCO_HIS, smos_files, "SMOS", Yorig=YORIG,
                                                  out_dir=VALIDATION_DIR)
    sat_stats_all["SMOS"] = smos_stats
    print(f"  -> {len(figs)} figure(s) written, one per day: "
         f"{[os.path.basename(p) for p in figs.values()]}")
    if smos_stats is not None and not smos_stats.empty:
        print("\nSMOS daily SSS statistics:")
        print(smos_stats.to_string(index=False))
        smos_stats.to_csv(os.path.join(VALIDATION_DIR, "sss_vs_smos_stats.csv"), index=False)

print()


  [SMOS] 2026-07-11:
  [SSS]  n=8070  bias=-0.086  RMSE=0.300  cRMSE=0.287  corr=0.588
  [SMOS] 2026-07-12:
  [SSS]  n=8070  bias=-0.088  RMSE=0.275  cRMSE=0.260  corr=0.602
  [SMOS] 2026-07-13:
  [SSS]  n=8070  bias=-0.081  RMSE=0.303  cRMSE=0.292  corr=0.457
  [SMOS] 2026-07-14:
  [SSS]  n=8070  bias=-0.073  RMSE=0.287  cRMSE=0.278  corr=0.526
  [SMOS] 2026-07-15:
  [SSS]  n=8070  bias=-0.095  RMSE=0.274  cRMSE=0.258  corr=0.570
  [SMOS] 2026-07-16:
  [SSS]  n=8070  bias=-0.047  RMSE=0.293  cRMSE=0.289  corr=0.611
  -> 6 figure(s) written, one per day: ['sss_vs_smos_2026-07-11.png', 'sss_vs_smos_2026-07-12.png', 'sss_vs_smos_2026-07-13.png', 'sss_vs_smos_2026-07-14.png', 'sss_vs_smos_2026-07-15.png', 'sss_vs_smos_2026-07-16.png']

SMOS daily SSS statistics:
     n      bias     rmse    crmse     corr  model_mean  ref_mean  model_min  model_max   ref_min   ref_max       date
8070.0 -0.086222 0.300048 0.287392 0.587630   36.186294 36.272516  34.047318  37.110340 34.923186 36.991510 202

## 8. In-situ validation -- trajectory & profile (Copernicus in-situ, class 4) 
## This section is optional, we may lack insitu data available around coastal region  

Everything above is "class 1/2" (model vs. an assimilative reanalysis/
reference product) -- the standard GODAE OceanView taxonomy also defines
"class 4": model vs. independent in-situ observations, which `sftools.
validation_godae.validate_against_insitu()` scores per GODAE depth layer
against CMEMS in-situ TAC profiles (https://doi.org/10.48670/moi-00036).

This is optional here (as it is in `sftools/run_validation.py` via
`--insitu-files`/`--require-insitu-pass`) because in-situ coverage for a
given cycle/region is often patchy -- a cycle with zero nearby profiles
isn't a validation failure, just a cycle with nothing to check here.

**Reference product:** CMEMS Global Ocean In-Situ NRT observations, `INSITU_GLO_PHYBGCWAV_DISCRETE_MYNRT_013_030` (DOI 10.48670/moi-00036); see the Product User Manual for the platform types, QC flags and file layout (CMEMS-INS-PUM-013-030-036) this reader assumes. Covers both **trajectory** platforms (drifting buoys, gliders -- a track of near-surface points over time, Section 8b) and **profile** platforms (Argo floats, CTD/XBT casts -- one vertical profile per station, scored in Section 8a and shown individually in Section 8c). Availability-checked (`AVAIL['insitu_nrt']`) and skipped if unavailable or if `INSITU_FILES` matches nothing for this cycle/region -- patchy in-situ coverage is normal, not a failure.


## Check the availability of in-situ product (skip)
**CMEMS in-situ TAC** -- one file per platform (trajectories, profiles & ARGO)

## Download in situ data (skip)

## Plot figures (skip)

## 9. HTML summary report

Everything above writes its own figures/CSVs straight into `VALIDATION_DIR`
as it goes -- useful while working through the notebook interactively, but
not something you'd want to browse file-by-file afterwards, or share as a
single artifact. This section gathers all of it into ONE self-contained
HTML page (`index.html`, no external dependencies, works offline): the
Section 5 pass/fail banner, then every figure grouped by section (bias
maps, scatter, Taylor diagrams, time series, boxplots, satellite SST/SSS,
in-situ), then every statistics table.

Nothing here re-computes anything -- it only reads whatever is already in
`VALIDATION_DIR` at this point, so re-running just this cell after e.g.
re-running one earlier section (a different `DEPTH_M`, say) refreshes the
page with the new figures. `validate_all_cycles.sh` (Section 10) calls this
same function after running each cycle, so every cycle ends up with the
same kind of report, browsable independently of the notebook.


In [74]:
html_path = val.build_html_summary(VALIDATION_DIR, cycle=CYCLE, config=CONFIG)
print(f"HTML summary written -> {html_path}")
print(f"Open it in a browser: file://{os.path.abspath(html_path)}") 


HTML summary written -> /home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260711/validation_cycle_20260711.html
Open it in a browser: file:///home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260711/validation_cycle_20260711.html


## 10. Batch validation across every forecast cycle -- using `validate_all_cycles.sh`

Everything above validates a *single* cycle, loaded interactively. In
production, every forecast cycle gets its own dated directory (e.g.
`forecast/model-runs/Canary_12/20260714/`) -- `forecast/validate_all_cycles.sh`
runs this exact notebook, plus the fast automated gate, across all of them.

**What it does, per cycle:**
1. `python -m sftools.run_validation` (Step 4.1) -- the fast, deterministic
   PASS/FAIL gate (GODAE scorecard, Section 9.3 criteria,
   `validation_report.json`/`.txt`, a combined Taylor diagram). This alone
   decides the script's exit code and each row of `validation_summary.csv`.
2. `jupyter nbconvert --execute` on **this notebook itself** (the cycle is
   selected via the `SEAFORWARD_CYCLE`/`SEAFORWARD_CONFIG`/`SEAFORWARD_MAIN_DIR`
   environment variables Section 0 already reads) -- so batch validation
   does *exactly* what this notebook does: satellite SST/SSS, in-situ,
   boxplots, one figure per day, and the HTML summary (Section 9) --
   without a second, separately-maintained implementation that could drift
   out of sync with this one. Informational only: its outcome doesn't
   affect the script's PASS/FAIL/exit code.

**Usage** (from a shell, in the `forecast/` directory):
```bash
./validate_all_cycles.sh                          # every config, every cycle
./validate_all_cycles.sh --config Canary_12        # one config only
./validate_all_cycles.sh --since 20260701          # skip cycles before this date
./validate_all_cycles.sh --force                   # re-validate everything
./validate_all_cycles.sh --dry-run                 # list what WOULD run, do nothing
./validate_all_cycles.sh --insitu-files "downloaded_data/INSITU/*.nc" --require-insitu-pass
                                                    # also gate on in-situ agreement
```

**Outputs, per config directory** (`<MAIN_DIR>/<CONFIG>/`):
- `<CYCLE>/fcst/validation/validation_report.json`/`.txt`/`taylor_diagram.png`
  -- from the fast gate (step 1).
- `validation_<CYCLE>/index.html` and every figure/CSV behind it -- from
  the full notebook run (step 2), same as Section 9 above.
- `validation_summary.csv` -- one row per cycle validated (fast-gate
  result + whether the notebook step succeeded), across every run of the
  script -- loaded and plotted below to show trends across cycles.

**Exit code:** `0` if every cycle run passed the fast gate (or nothing
needed running); `1` if at least one FAILED or ERRORED -- meaningful from
cron without reading the log.


---
## Notes

- **DCC linkage (FR-12):** this notebook implements process **V1**
  (Verification & Analysis layer). Its inputs come from **C1** (the CROCO
  run) and its outputs (validated CROCO fields, this pass/fail summary)
  feed the **D1** notebooks (`01_seaforward_postprocess_plot.ipynb`,
  `04_exercises.ipynb`, `05_sensitivity.ipynb`, `06_animation.ipynb`),
  as well as the multi-cycle composite validation in
  `03_composite_validation.ipynb`.
- **QA (Testing and Validation Plan Section 9.1):** this notebook is
  designed to execute without errors from a fresh kernel restart +
  run-all against a real forecast cycle (see Section 1).
- **Next:** `04_exercises.ipynb` for guided, hands-on diagnostics
  (upwelling index, mixed-layer depth, coastal jet, eddy detection) built
  on the same CROCO output.
